# Modulo di Analisi Ambientale — Marine AI

Costruisce, per le specie con copertura tassonomica affidabile (`env_module_ready = True`, notebook 02), tre analisi: distribuzione geografica, trend temporale delle osservazioni, rilevamento di anomalie anno su anno.

**Scelta di scoping:** alcune specie hanno centinaia di migliaia di occorrenze OBIS (es. l'aragosta americana ne supera le 900.000). Scaricare l'intero storico per tutte le 145 specie non e' necessario per le analisi qui presentate: si lavora su un campione per specie (fino a `MAX_RECORDS_PER_SPECIES`), sufficiente per stimare distribuzione e trend senza sovraccaricare l'API pubblica di OBIS.

**Nota implementativa:** il codice segue la documentazione ufficiale dell'API OBIS v3 ma richiede un ambiente con accesso di rete per essere eseguito.

## 0. Setup

In [1]:
# pip install requests pandas numpy plotly --quiet

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.express as px
from tqdm.auto import tqdm

def find_project_root(marker=".gitignore"):
    path = Path.cwd()
    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent
    raise FileNotFoundError(f"Marker '{marker}' non trovato risalendo da {Path.cwd()}")

PROJECT_ROOT = find_project_root()
COVERAGE_CSV = PROJECT_ROOT / "docs" / "species_environmental_coverage.csv"
OUTPUT_DIR = PROJECT_ROOT / "docs"
OUTPUT_DIR.mkdir(exist_ok=True)

OBIS_BASE = "https://api.obis.org/v3"
SESSION = requests.Session()
SESSION.headers.update({"Accept": "application/json"})
REQUEST_DELAY = 0.3
MAX_RECORDS_PER_SPECIES = 3000

print("Project root:", PROJECT_ROOT)
print("Coverage CSV:", COVERAGE_CSV)

Project root: /home/ubuntu/marine_ai
Coverage CSV: /home/ubuntu/marine_ai/docs/species_environmental_coverage.csv


## 1. Elenco specie idonee al modulo ambientale

In [2]:
coverage_df = pd.read_csv(COVERAGE_CSV)

ready_df = coverage_df[coverage_df["env_module_ready"] == True].drop_duplicates(subset="aphia_id").copy()
ready_df["aphia_id"] = ready_df["aphia_id"].astype(int)

print(f"Specie uniche (per AphiaID) pronte per l'analisi ambientale: {len(ready_df)}")
ready_df[["canonical_label", "aphia_id", "obis_occurrence_count"]].sort_values("obis_occurrence_count", ascending=False).head(10)

Specie uniche (per AphiaID) pronte per l'analisi ambientale: 141


,canonical_label,aphia_id,obis_occurrence_count
174,american lobster,156134,921016.0
131,eudyptula minor,225932,557895.0
40,strongylocentrotus franciscanus,240762,378611.0
42,green sea turtle,137206,243974.0
50,bull shark,105792,236322.0
127,blacktip reef shark,105795,226964.0
187,pygoscelis adeliae,225757,189836.0
138,atlantic blue crab,107379,189187.0
62,european green crab,107381,188042.0
192,sea turtle,137205,173590.0


## 2. Download campione di occorrenze geolocalizzate

Per ogni specie, richiediamo solo i campi necessari (coordinate, anno, profondita'), fino a `MAX_RECORDS_PER_SPECIES` record.

In [3]:
def fetch_occurrence_sample(aphia_id, size=MAX_RECORDS_PER_SPECIES):
    url = f"{OBIS_BASE}/occurrence"
    params = {
        "taxonid": aphia_id,
        "size": size,
        "fields": "decimalLongitude,decimalLatitude,date_year,depth",
    }
    try:
        resp = SESSION.get(url, params=params, timeout=30)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except requests.RequestException as e:
        print(f"Errore per AphiaID {aphia_id}: {e}")
        return []

all_occurrences = []
for _, row in tqdm(ready_df.iterrows(), total=len(ready_df)):
    records = fetch_occurrence_sample(row["aphia_id"])
    for r in records:
        r["canonical_label"] = row["canonical_label"]
        r["aphia_id"] = row["aphia_id"]
    all_occurrences.extend(records)
    time.sleep(REQUEST_DELAY)

occ_df = pd.DataFrame(all_occurrences)
occ_df = occ_df.dropna(subset=["decimalLongitude", "decimalLatitude"])
print(f"Record scaricati: {len(occ_df)} (su {len(ready_df)} specie)")
occ_df.head(3)

  0%|          | 0/141 [00:00<?, ?it/s]

Record scaricati: 260477 (su 141 specie)


,decimalLatitude,decimalLongitude,canonical_label,aphia_id,date_year,depth
0,-18.766667,147.516667,fish,219872,NaN,NaN
1,-20.040830,117.806670,fish,219872,1983.0,40.0
2,-20.040830,117.806670,fish,219872,1983.0,40.0


In [4]:
occ_df.to_csv(OUTPUT_DIR / "obis_occurrence_sample.csv", index=False)
print("Salvato obis_occurrence_sample.csv — riusabile dal modulo RAG per rispondere a domande di distribuzione")

Salvato obis_occurrence_sample.csv — riusabile dal modulo RAG per rispondere a domande di distribuzione


## 3. Distribuzione geografica

Mappa globale del campione raccolto, e vista di dettaglio su una specie a scelta (di default la green sea turtle, gia' discussa nel deck di presentazione).

In [5]:
fig = px.scatter_geo(
    occ_df.sample(min(20000, len(occ_df)), random_state=42),
    lat="decimalLatitude", lon="decimalLongitude",
    opacity=0.35, projection="natural earth",
    title="Distribuzione campionata di tutte le specie coperte (OBIS)",
)
fig.update_traces(marker=dict(size=3, color="#065A82"))
fig.show()

In [6]:
FOCUS_SPECIES = "green sea turtle"  # cambia con qualunque canonical_label presente in ready_df

focus_df = occ_df[occ_df["canonical_label"] == FOCUS_SPECIES]
print(f"Occorrenze campionate per '{FOCUS_SPECIES}': {len(focus_df)}")

fig = px.scatter_geo(
    focus_df, lat="decimalLatitude", lon="decimalLongitude",
    projection="natural earth", title=f"Distribuzione — {FOCUS_SPECIES}",
)
fig.update_traces(marker=dict(size=4, color="#F4A261"))
fig.show()

Occorrenze campionate per 'green sea turtle': 3000


## 4. Trend temporale

**Avvertenza metodologica, da riportare in sinossi/report:** un aumento delle osservazioni anno su anno non implica necessariamente un aumento della popolazione — puo' riflettere un aumento dello sforzo di campionamento (piu' ricerca, piu' citizen science, piu' dataset digitalizzati) tanto quanto un cambiamento ecologico reale. Il trend qui calcolato descrive l'andamento delle osservazioni registrate in OBIS, non necessariamente l'andamento della popolazione della specie.

In [7]:
def yearly_counts(df, min_year=1950):
    df = df.dropna(subset=["date_year"]).copy()
    df["date_year"] = df["date_year"].astype(int)
    df = df[df["date_year"] >= min_year]
    counts = df.groupby("date_year").size().reset_index(name="n_occurrences")
    return counts

yearly = yearly_counts(focus_df)

fig = px.line(yearly, x="date_year", y="n_occurrences", markers=True,
              title=f"Osservazioni per anno (campione) — {FOCUS_SPECIES}")
fig.update_traces(line_color="#065A82")
fig.show()

if len(yearly) >= 3:
    slope, intercept = np.polyfit(yearly["date_year"], yearly["n_occurrences"], 1)
    direction = "in aumento" if slope > 0 else "in diminuzione"
    print(f"Trend lineare del campione: {direction} di circa {abs(slope):.1f} osservazioni/anno")

Trend lineare del campione: in aumento di circa 5.0 osservazioni/anno


## 5. Rilevamento anomalie

Metodo semplice e trasparente: z-score sulle osservazioni annuali rispetto a media e deviazione standard della serie. Anni con `|z| > 2` sono segnalati come anomali — non necessariamente un evento biologico reale, ma un punto che merita approfondimento (cambio nello sforzo di campionamento, evento migratorio, mass stranding, ecc.).

In [8]:
def detect_anomalies(yearly_df, z_threshold=2.0):
    yearly_df = yearly_df.copy()
    mean, std = yearly_df["n_occurrences"].mean(), yearly_df["n_occurrences"].std()
    yearly_df["z_score"] = (yearly_df["n_occurrences"] - mean) / std if std > 0 else 0
    yearly_df["is_anomaly"] = yearly_df["z_score"].abs() > z_threshold
    return yearly_df

yearly_with_anomalies = detect_anomalies(yearly)
anomalies = yearly_with_anomalies[yearly_with_anomalies["is_anomaly"]]

print(f"Anni anomali rilevati per {FOCUS_SPECIES}: {len(anomalies)}")
anomalies

Anni anomali rilevati per green sea turtle: 2


,date_year,n_occurrences,z_score,is_anomaly
23,2013,487,3.307526,True
28,2018,381,2.445190,True


## 6. Esecuzione su tutte le specie ed export

Ripetiamo trend + anomalie per ciascuna delle specie disponibili, cosi' il modulo RAG puo' rispondere a domande su qualunque specie del classificatore, non solo su quella scelta come esempio sopra.

In [9]:
all_yearly = []
all_anomalies = []

for label in tqdm(occ_df["canonical_label"].unique()):
    species_df_subset = occ_df[occ_df["canonical_label"] == label]
    yc = yearly_counts(species_df_subset)
    if len(yc) < 3:
        continue  # serie troppo corta per un trend/anomalia sensati
    yc_anom = detect_anomalies(yc)
    yc_anom["canonical_label"] = label
    all_yearly.append(yc_anom)
    anomalous_years = yc_anom[yc_anom["is_anomaly"]]
    if len(anomalous_years) > 0:
        all_anomalies.append(anomalous_years)

yearly_all_df = pd.concat(all_yearly, ignore_index=True) if all_yearly else pd.DataFrame()
anomalies_all_df = pd.concat(all_anomalies, ignore_index=True) if all_anomalies else pd.DataFrame()

yearly_all_df.to_csv(OUTPUT_DIR / "species_yearly_trends.csv", index=False)
anomalies_all_df.to_csv(OUTPUT_DIR / "species_anomalies.csv", index=False)

print(f"Specie con serie temporale sufficiente per l'analisi: {yearly_all_df['canonical_label'].nunique()}")
print(f"Totale anni anomali rilevati (tutte le specie): {len(anomalies_all_df)}")
print("Salvati species_yearly_trends.csv e species_anomalies.csv in docs/")

  0%|          | 0/141 [00:00<?, ?it/s]

Specie con serie temporale sufficiente per l'analisi: 139
Totale anni anomali rilevati (tutte le specie): 353
Salvati species_yearly_trends.csv e species_anomalies.csv in docs/


**Limite noto del metodo, verificato sui dati:** circa il 30% delle anomalie rilevate si concentra nel triennio 2023-2025, distribuito su decine di specie senza relazione ecologica tra loro. Il pattern e' coerente con una crescita generale della piattaforma OBIS (piu' digitalizzazione, piu' citizen science) negli anni recenti, non con un fenomeno biologico condiviso. Il rilevamento anomalie attuale opera sul conteggio grezzo di osservazioni per anno e non compensa questo trend di fondo; uno sviluppo futuro naturale e' applicare lo z-score ai residui rispetto a un trend (o al tasso di crescita anno su anno) invece che al valore assoluto.